# 04. 시장과 목적지의 수요 패턴

2023–2025년 관측 여객으로 시장 흐름을 확인하고, 기존 인기 목적지와 최근 성장 목적지를 구분한다. 증가 인원과 증가율을 함께 본 뒤 두 집단의 월별 수요 패턴을 비교한다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from sqlalchemy import text
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'scripts'))
import portfolio_data as pdx
engine=pdx.get_engine()
business=pdx.named_sql('04_business_metrics.sql')
pdx.apply_ddl(engine,business,['create_v_metric_demand_destination_monthly'])
def query(name):return pdx.query(engine,business,name)
PLOTLY_TEMPLATE='plotly_white'
def chart_title(title,note):return f'{title}<br><sup>2023–2025 · {note}</sup>'
parity=query('demand_metric_monthly_total_check')
assert parity.checked_months.iat[0]==36 and parity.passenger_absolute_difference.iat[0]==0
display(parity.head())

,checked_months,passenger_absolute_difference,passenger_max_month_difference
0,36,0.0,0.0


## 1. 가설 1 — 누적 여객이 많은 목적지는 최근에도 큰 수요를 보이겠지만, 최근 증가 인원 상위에는 누적 인기 상위권 밖의 목적지도 포함될 것이다.

2023–2025년 누적 여객 상위 10곳이 2025년 수요 상위에도 남아 있는지 확인한다. 최근 변화는 2024→2025년 증가 <strong>인원</strong>으로 순위를 매기고, <strong>증가율</strong>과 2024년 기준 수요도 함께 보아 작은 시장의 높은 비율을 구분한다.

In [2]:
market=query('monthly_demand_metrics')
annual=(market.assign(year=market.period_ym.astype(str).str[:4])
        .groupby('year',as_index=False)['passengers'].sum())
display(market.head())
display(annual)

,period_ym,passengers,operations,passengers_per_operation,previous_year_passengers,passenger_yoy_rate,previous_year_operations,operation_yoy_rate
0,202301,2403433.0,13632.0,176.31,NaN,NaN,NaN,NaN
1,202302,2212589.0,12937.0,171.03,NaN,NaN,NaN,NaN
2,202303,2256529.0,14407.0,156.63,NaN,NaN,NaN,NaN
3,202304,2412373.0,15231.0,158.39,NaN,NaN,NaN,NaN
4,202305,2622755.0,16781.0,156.29,NaN,NaN,NaN,NaN


,year,passengers
0,2023,34141927.0
1,2024,44410715.0
2,2025,47221285.0


In [3]:
fig=px.line(market.assign(month=pd.to_datetime(market.period_ym,format='%Y%m')),
            x='month',y='passengers',markers=True,
            title=chart_title('월별 국제선 출발 여객 추이','전국 노선 여객'),
            labels={'month':'연월','passengers':'여객 수'},template=PLOTLY_TEMPLATE)
fig.update_yaxes(tickformat=',')
fig.update_layout(height=450,showlegend=False)
fig.show()

In [4]:
yearly=query('yearly_destination_demand_metrics')
growth=query('growth_destination_metrics')
display(yearly.head())
display(growth.head())

dest=pd.read_sql(text('SELECT period_ym,destination_canonical_std AS dest,passengers FROM v_metric_demand_destination_monthly'),engine)
dest['year']=dest.period_ym.str[:4].astype(int)
cum=dest.groupby('dest').passengers.sum().nlargest(10)
y=dest.groupby(['dest','year']).passengers.sum().unstack(fill_value=0)
recent=y[2025].nlargest(10)
inc=(y[2025]-y[2024]).nlargest(10)
growth_top10=pd.DataFrame({
    'dest':inc.index,
    'passengers_2024':y.loc[inc.index,2024].to_numpy(),
    'passengers_2025':y.loc[inc.index,2025].to_numpy(),
    'increase':inc.to_numpy(),
})
growth_top10['rate_pct']=100*growth_top10.increase/growth_top10.passengers_2024
growth_top10['rate_label']=growth_top10.rate_pct.map(lambda v:f'{v:.1f}%')
display(growth_top10[['dest','passengers_2024','passengers_2025','increase','rate_pct']].head())

coverage_24_25=(dest[dest.year.isin([2024,2025])]
                .groupby(['dest','year']).period_ym.nunique().unstack(fill_value=0))
full_year=coverage_24_25.index[(coverage_24_25[2024]==12)&(coverage_24_25[2025]==12)]
rate_view=pd.DataFrame({
    'dest':full_year,
    'passengers_2024':y.loc[full_year,2024].to_numpy(),
    'passengers_2025':y.loc[full_year,2025].to_numpy(),
})
rate_view=rate_view[rate_view.passengers_2024>0].copy()
rate_view['increase']=rate_view.passengers_2025-rate_view.passengers_2024
rate_view['rate_pct']=100*rate_view.increase/rate_view.passengers_2024
rate_top10=rate_view.nlargest(10,'rate_pct').copy()
rate_top10['rate_label']=rate_top10.rate_pct.map(lambda v:f'{v:.1f}%')
print('두 연도 모두 12개월 관측한 목적지의 증가율 상위 5곳:')
display(rate_top10[['dest','passengers_2024','passengers_2025','increase','rate_pct']].head().round({'rate_pct':1}))

recent_overlap=set(recent.index)&set(cum.index)
outside=[d for d in inc.index if d not in cum.index]
print('누적 상위 10 중 2025년 수요 상위 10:',len(recent_overlap))
print('누적 인기 상위 10 밖의 증가 인원 상위 목적지:',outside)
assert len(recent_overlap)==9 and len(outside)==6

,operating_year,total_rank,total_period_passengers,destination_canonical_std,passengers,operations,passenger_share,passengers_per_operation,previous_annual_passengers,annual_passenger_yoy_rate,previous_annual_operations,annual_operation_yoy_rate
0,2023,1,10074906.0,간사이,2958140.0,15827.0,0.0866,186.90,NaN,NaN,NaN,NaN
1,2024,1,10074906.0,간사이,3499844.0,18606.0,0.0788,188.10,2958140.0,0.1831,15827.0,0.1756
2,2025,1,10074906.0,간사이,3616922.0,20287.0,0.0766,178.29,3499844.0,0.0335,18606.0,0.0903
3,2023,2,8028573.0,도쿄나리타,2193165.0,12690.0,0.0642,172.83,NaN,NaN,NaN,NaN
4,2024,2,8028573.0,도쿄나리타,2929036.0,16181.0,0.0660,181.02,2193165.0,0.3355,12690.0,0.2751


,destination_canonical_std,latest_year,previous_year,latest_year_observed_months,previous_year_observed_months,latest_year_passengers,comparison_year_passengers,latest_year_operations,comparison_year_operations,passenger_growth,passenger_growth_rate,operation_growth,operation_growth_rate
0,푸동,2025,2024,12,12,1865525.0,1452818.0,11849.0,10299.0,412707.0,0.2841,1550.0,0.1505
1,타이페이,2025,2024,12,12,2194821.0,1995187.0,10419.0,9691.0,199634.0,0.1001,728.0,0.0751
2,홍콩,2025,2024,12,12,1626343.0,1454544.0,9416.0,9269.0,171799.0,0.1181,147.0,0.0159
3,청도,2025,2024,12,12,910626.0,745852.0,6521.0,5813.0,164774.0,0.2209,708.0,0.1218
4,나고야,2025,2024,12,12,716572.0,564569.0,4147.0,3333.0,152003.0,0.2692,814.0,0.2442


,dest,passengers_2024,passengers_2025,increase,rate_pct
0,푸동,1452818.0,1865525.0,412707.0,28.407344
1,타이페이,1995187.0,2194821.0,199634.0,10.005779
2,홍콩,1454544.0,1626343.0,171799.0,11.811193
3,청도,745852.0,910626.0,164774.0,22.092050
4,나고야,564569.0,716572.0,152003.0,26.923724


두 연도 모두 12개월 관측한 목적지의 증가율 상위 5곳:


,dest,passengers_2024,passengers_2025,increase,rate_pct
5,구마모도,65019.0,160019.0,95000.0,146.1
123,쿤밍,19621.0,47905.0,28284.0,144.2
26,따이쭝,68490.0,157592.0,89102.0,130.1
40,멕시코,31398.0,69143.0,37745.0,120.2
116,카고시마,37969.0,74390.0,36421.0,95.9


누적 상위 10 중 2025년 수요 상위 10: 9
누적 인기 상위 10 밖의 증가 인원 상위 목적지: ['청도', '나고야', '오끼나와', '북경', '센젠', '카오슝']


In [5]:
fig=px.bar(x=cum.index,y=cum.values,
           title=chart_title('3개년 누적 인기 목적지','여객 수 상위 10'),
           labels={'x':'목적지','y':'누적 여객 수'},template=PLOTLY_TEMPLATE)
fig.update_yaxes(tickformat=',')
fig.show()

In [6]:
fig=px.bar(growth_top10,x='dest',y='increase',text='rate_label',
           custom_data=['passengers_2024','passengers_2025','rate_pct'],
           title=chart_title('2024→2025 증가 인원 상위 목적지','막대=증가 인원 · 숫자=증가율'),
           labels={'dest':'목적지','increase':'증가 인원'},template=PLOTLY_TEMPLATE)
fig.update_traces(textposition='outside',cliponaxis=False,
                  hovertemplate='목적지=%{x}<br>2024년=%{customdata[0]:,.0f}명<br>2025년=%{customdata[1]:,.0f}명<br>증가 인원=%{y:,.0f}명<br>증가율=%{customdata[2]:.1f}%<extra></extra>')
fig.update_yaxes(tickformat=',')
fig.update_layout(height=480,margin=dict(t=90))
fig.show()

In [7]:
fig=px.bar(rate_top10,x='dest',y='rate_pct',text='rate_label',
           custom_data=['passengers_2024','passengers_2025','increase'],
           title=chart_title('2024→2025 증가율 상위 목적지','두 연도 12개월 관측 · 분모=2024년 여객'),
           labels={'dest':'목적지','rate_pct':'증가율 (%)'},template=PLOTLY_TEMPLATE)
fig.update_traces(textposition='outside',cliponaxis=False,
                  hovertemplate='목적지=%{x}<br>2024년=%{customdata[0]:,.0f}명<br>2025년=%{customdata[1]:,.0f}명<br>증가 인원=%{customdata[2]:,.0f}명<br>증가율=%{y:.1f}%<extra></extra>')
fig.update_layout(height=480,margin=dict(t=90))
fig.show()

### 가설 1 결과와 판단

<strong>관측 결과는 가설 1에서 예상한 방향과 같다.</strong> 누적 인기 상위 10곳 중 9곳은 2025년 수요 상위에도 포함된다. 최근 증가 인원 상위 10곳 중 6곳은 누적 인기권 밖이다. 따라서 큰 수요가 유지되는 목적지와 최근 수요가 빠르게 커진 목적지를 한 순위로만 고르기 어렵다.

증가율 순위는 규모가 작은 목적지를 더 잘 드러낸다. 구마모토는 증가율 146.1%에 증가 인원 약 9.5만 명, 쿤밍은 144.2%에 약 2.8만 명이다. 푸둥은 증가율이 28.4%여도 약 41.3만 명 늘었다. 프로모션 검토 시 기존 인기 목적지는 수요 규모를, 최근 성장 목적지는 증가 인원·증가율과 기준 수요를 함께 확인한다. 이는 관측 수요의 변화이며 미래 수요나 캠페인 성과는 아니다.

## 2. 가설 2 — 목적지별로 과거 수요가 가장 많은 달은 서로 다를 것이다.

각 연도의 월평균을 1로 놓은 계절지수로 월별 상대 수요를 비교한다. 가설 1에서 본 누적 인기, 증가 인원, 증가율 기준의 목적지를 각각 5곳씩 골라 월별 패턴을 확인한다.

In [8]:
overall=pdx.query(engine,pdx.named_sql('03_eda.sql'),'seasonal_demand_profile')
season=query('seasonal_destination_metrics')
display(overall[['calendar_month','avg_passenger_seasonal_index']].head())
display(season.head())

market_ym=pd.read_sql(text('SELECT period_ym,SUM(passengers) AS passengers FROM v_route_monthly WHERE is_international=TRUE GROUP BY period_ym'),engine)
market_ym['year']=market_ym.period_ym.str[:4].astype(int)
market_ym['month']=market_ym.period_ym.str[-2:].astype(int)
market_ym['index']=market_ym.passengers/market_ym.groupby('year').passengers.transform('mean')
market_index=market_ym.groupby('month')['index'].mean()

popular=list(cum.head(5).index)
rising=outside[:5]
rate_rising=rate_top10.head(5).dest.tolist()
selected=list(dict.fromkeys(popular+rising+rate_rising))
selected_monthly=dest[dest.dest.isin(selected)].copy()
selected_monthly['month']=selected_monthly.period_ym.str[-2:].astype(int)
selected_monthly['observed_months']=(selected_monthly.groupby(['dest','year'])
                                     .month.transform('nunique'))
selected_monthly=selected_monthly[selected_monthly.observed_months==12].copy()
complete_years=selected_monthly.groupby('dest').year.nunique().reindex(selected)
assert len(popular)==len(rising)==len(rate_rising)==5 and complete_years.ge(2).all()
selected_monthly['index']=(selected_monthly.passengers/
                           selected_monthly.groupby(['dest','year']).passengers.transform('mean'))
seasonal_15=selected_monthly.groupby(['dest','month'],as_index=False)['index'].mean()
heat_all=seasonal_15.pivot(index='dest',columns='month',values='index').loc[selected]
popular_heat=heat_all.loc[popular]
rising_heat=heat_all.loc[rising]
rate_heat=heat_all.loc[rate_rising]
color_min=max(0,float(heat_all.min().min())-0.03)
color_max=float(heat_all.max().max())+0.03
neutral=(1-color_min)/(color_max-color_min)
heat_colors=[[0,'#2768A2'],[neutral,'#F8FAFC'],[1,'#B44949']]

def seasonal_heatmap(matrix,title):
    fig=px.imshow(matrix,aspect='auto',text_auto='.2f',
                  color_continuous_scale=heat_colors,
                  range_color=(color_min,color_max),
                  labels={'x':'월','y':'목적지','color':'계절지수'},
                  title=chart_title(title,'완전 관측 연도 · 1=연도별 월평균'),
                  template=PLOTLY_TEMPLATE)
    fig.update_traces(hovertemplate='목적지=%{y}<br>월=%{x}월<br>계절지수=%{z:.2f}<extra></extra>')
    fig.update_xaxes(dtick=1,side='top')
    fig.update_layout(height=460,margin=dict(l=100,t=90))
    return fig

peak=(seasonal_15.sort_values(['dest','index'],ascending=[True,False])
      .drop_duplicates('dest').set_index('dest'))
peak_by_group=pd.concat([
    peak.loc[popular].assign(group='기존 인기'),
    peak.loc[rising].assign(group='증가 인원'),
    peak.loc[rate_rising].assign(group='증가율'),
]).reset_index()
print('전체 시장 최고월:',int(market_index.idxmax()))
print('계절지수에 사용한 완전 관측 연도 수:',complete_years.to_dict())
print('집단별 최고월 분포:')
display(pd.crosstab(peak_by_group.group,peak_by_group.month))
display(peak_by_group.loc[peak_by_group.month!=int(market_index.idxmax()),['group','dest','month','index']])
assert peak_by_group.month.value_counts().sort_index().to_dict()=={1:1,8:2,10:2,12:10}

,calendar_month,avg_passenger_seasonal_index
0,01,0.9780
1,02,0.8879
2,03,0.8749
3,04,0.9081
4,05,0.9492


,calendar_month,destination_canonical_std,observed_years,avg_passengers,min_passengers,max_passengers,avg_passenger_seasonal_index,avg_operations,demand_rank
0,01,간사이,3,275182.0,201845.0,322454.0,0.9738,1434.0,1
1,01,도쿄나리타,3,214936.0,148676.0,263036.0,0.9515,1203.0,2
2,01,후쿠오카,3,189172.0,161885.0,208598.0,0.9953,977.0,3
3,01,방콕,3,174275.0,165078.0,188576.0,1.2531,689.0,4
4,01,타이페이,3,142045.0,79503.0,181915.0,0.8774,706.0,5


전체 시장 최고월: 12
계절지수에 사용한 완전 관측 연도 수: {'간사이': 3, '도쿄나리타': 3, '후쿠오카': 3, '타이페이': 3, '방콕': 3, '청도': 3, '나고야': 3, '오끼나와': 3, '북경': 3, '센젠': 3, '구마모도': 3, '쿤밍': 2, '따이쭝': 2, '멕시코': 3, '카고시마': 2}
집단별 최고월 분포:


month,1,8,10,12
group,,,,
기존 인기,1,0,0,4
증가 인원,0,1,1,3
증가율,0,1,1,3


,group,dest,month,index
4,기존 인기,방콕,1,1.253111
6,증가 인원,나고야,10,1.249287
8,증가 인원,북경,8,1.383150
11,증가율,쿤밍,10,1.433394
13,증가율,멕시코,8,1.811240


In [9]:
fig=px.line(x=market_index.index,y=market_index.values,markers=True,
            title=chart_title('전체 국제선 월별 계절지수','각 연도 월평균 대비'),
            labels={'x':'월','y':'평균 계절지수'},template=PLOTLY_TEMPLATE)
fig.add_hline(y=1,line_dash='dash',line_color='#64748B')
fig.update_xaxes(dtick=1)
fig.show()

### 히트맵 대상 선정

- <strong>기존 인기 5곳</strong>: 가설 1의 2023–2025년 누적 여객 상위 10곳에서 앞의 5곳.
- <strong>증가 인원 5곳</strong>: 2024→2025년 증가 인원 상위 10곳 중 누적 인기 상위 10곳과 겹치지 않는 목적지에서 앞의 5곳.
- <strong>증가율 5곳</strong>: 2024·2025년을 모두 12개월 관측한 목적지의 증가율 상위 5곳. 증가 인원 순위와는 별개다.

계절지수는 2023–2025년 중 목적지별로 12개월이 모두 관측된 연도만 평균한다. 따라서 2023년 기록이 짧은 증가율 상위 목적지는 2024–2025년 자료로 계산한다.

In [10]:
fig=seasonal_heatmap(popular_heat,'누적 여객 상위 5곳의 월별 계절지수')
fig.show()

In [11]:
fig=seasonal_heatmap(rising_heat,'누적 인기권 밖 증가 인원 상위 5곳의 월별 계절지수')
fig.show()

In [12]:
fig=seasonal_heatmap(rate_heat,'2024→2025 증가율 상위 5곳의 월별 계절지수')
fig.show()

### 가설 2 결과와 판단

<strong>관측 결과는 가설 2에서 예상한 방향과 같다.</strong> 전체 시장은 12월에 상대 수요가 가장 높다. 기존 인기 상위 5곳 중 4곳도 12월이지만 방콕은 1월이다. 증가 인원 상위 5곳에서는 3곳이 12월, 북경은 8월, 나고야는 10월이 최고월이다. 증가율 상위 5곳도 3곳은 12월이지만 멕시코는 8월, 쿤밍은 10월에 가장 높다.

시장 전체의 성수월 하나를 모든 목적지에 적용할 수 없다. 특히 증가율 상위에는 2023년 기록이 짧은 목적지가 있어 계절지수는 12개월이 모두 관측된 연도만 평균했다. 히트맵의 `성수월`은 과거 관측 수요의 계절지수 기준이며, 미래 예측 수요가 가장 큰 `추천월` 및 실제 캠페인 집행 시점과 다르다.

## 가설별 결론 종합

| 가설 | 관측 결과 | 프로모션 후보 검토에 주는 의미 |
|---|---|---|
| 1. 기존 인기와 최근 성장 | 누적 인기 상위 10곳 중 9곳은 2025년 수요 상위에도 있지만, 증가 인원 상위 10곳 중 6곳은 누적 인기권 밖이다. 증가율 상위에는 기준 수요가 작은 목적지도 나타난다. | 기존 인기 목적지는 수요 규모를, 최근 성장 목적지는 증가 인원·증가율·기준 수요를 함께 본다. |
| 2. 목적지별 성수월 | 세 집단 모두 12월이 가장 많은 목적지의 최고월이지만, 방콕은 1월, 북경·멕시코는 8월, 나고야·쿤밍은 10월에 높다. | 전체 시장의 성수월을 일괄 적용하지 않고 목적지별 월 패턴을 확인한다. |

두 가설 모두 관측 자료에서 예상한 방향과 같다. 여기서 확인한 것은 <strong>과거 수요의 규모·변화·시기</strong>다. 미래 예측 수요와 실제 프로모션 성과는 별도로 판단한다.